# Rough Implementation of Blueprint
 This notebook contains a rough implementation of the Blueprint. It is used to:
 - **Verify** if the concept works in reality
 - **Brainstorm** different **approaches** for each solution
- Identify **edge cases** in the solution.

Here in our demo we will use the **"Qwen 3.5 9B Q4 K M" (in GGUF format)** Open Source model as the central control brain.
And to use that LLM we will use **Llama.cpp (Python Library)**.

In [3]:
from llama_cpp import Llama, LlamaGrammar
# Llama grammar is required for forcing the output to be in our expected format.

We are **simulating** the industry server on our local computer.

Since on my **Mac** model is located in the downloads folder `/Users/noyan/downloads/Qwen3.5-9B-Q4_K_M.gguf`

For real work it needs alot more context to breath, but for testing purpose I am keeping it small `2000`, so that it consumes less RAM.

In [4]:
qwen = Llama(
    model_path = "/Users/noyan/downloads/Qwen3.5-9B-Q4_K_M.gguf",
    n_ctx = 2000,
    verbose = False # For agentic task we don't need verbose responses
)
print("Model Loaded Successfully!!")

output = qwen.create_chat_completion(
    messages=[{"role": "system", "content": "Hello How can I help you?"},
              {"role": "user", "content": "Hello"}],
    temperature=0,
    max_tokens=1000,
)
print(output)

Model Loaded Successfully!!
{'id': 'chatcmpl-6ffdb351-73d0-4891-86be-e68a1cab1b8c', 'object': 'chat.completion', 'created': 1790189857, 'model': '/Users/noyan/downloads/Qwen3.5-9B-Q4_K_M.gguf', 'choices': [{'index': 0, 'message': {'role': 'assistant', 'content': 'Hello! How can I help you today? Feel free to ask me questions, need advice, or just want to chat.'}, 'logprobs': None, 'finish_reason': 'stop'}], 'usage': {'prompt_tokens': 25, 'completion_tokens': 25, 'total_tokens': 50}}


In [5]:
completion = output["choices"][0]["message"]["content"]
print(completion)

Hello! How can I help you today? Feel free to ask me questions, need advice, or just want to chat.


For now, we first need to brainstorm on the capabilities of the agent and design the execution for each of the task.
Capabilities:
1. Read/Write Files
2. List files, can create directories
3. Fetch the metadata for each file.
4. Managing git like log and version control, where each commit is trackable.
5. Run command in the sandbox, code execution (Python or JavaScript or both)

We will store the relative path of the workspace of the agent in the `rel_path` variable.
We will use `pathlib` to deal with paths.

- `.resolve()` creates absolute path from relative path, with respect to working directory
- `Path()` converts the input into a path variable.
- `.mkdir(exist_ok = True, parents = True)` It creates directory at the given location, `exist_ok = True` ensures no error occurs if the directory already exists, `parents = True` creates any parent directory if missing instead of throwing errors.

In [6]:
from pathlib import Path
rel_path = Path('./workspace').resolve()
print(rel_path)
rel_path.mkdir(exist_ok = True, parents = True)

/Users/noyan/Desktop/Smart India Hackathon/AgenticAI/Blueprint/workspace


Since we are simulating the project here, I will create a directory called `LocalStorage`, which behaves as the local data base of the industry.

The **central_backend.py** will act as a **Backbone** for the project.

The **sandbox.py** will help in code execution by **LLM**.

The **Interpreter** directory contains interpreters for different types of files.

The **LLM** directory will contain the open-source models in gguf format.

The **cli.py** will be the Command Line Interface Application to use the agent.

The **GUI** directory will be the Graphical User Interface Application to use the Agent.

Since **Central Backend** is the _spinal cord_ of this project, which controls the entire data flow in the project, it needs to be build first.

The core algorithm behind **Central Backend** is:
1. Take **User's Input**.
2. **Feed** it to LLM with the prompt.
3. Proceed for task created by **LLM**.
4. **Feed** the output of task, along with the previous context to the LLM.
5. Get the new tasks and **repeat** from step 3, until all tasks are finished.
6. Show the **final inference** of the session.

**Github Issue 1:** _Chat Completion method of Agents_
Solution Algorithm (Simplified Version):
1. Take **User's Input**.
2. **Feed** it to LLM with the prompt and perform reasoning task on user's input and identify which tasks are independent and can be done simultaneously to get the idea of the workspace.
3. Get the **JSON** sequence of tasks and Proceed for those tasks **simultaneously**.
4. **Feed** the output of task, along with the previous context to the LLM and perform reasoning task on context and output and identify which tasks are independent and can be done simultaneously to get the idea of the workspace.
5. Get the new tasks and **repeat** from step 3, until all tasks are finished.
6. Show the **final inference** of the session.

**Github Issue 3:** _Inefficient file search by Central Backend_
Solution Algorithm (Simplified Version):
1. Take **User's Input**.
2. **Feed** it to LLM with the prompt and perform reasoning task on user's input and identify which tasks are independent and can be done simultaneously to get the idea of the workspace.
3. Get the **JSON** sequence of tasks and Proceed for those tasks **simultaneously**, and for file reading tasks to get just the context of each file, make use of LLM prepared one-time meta-data.
4. **Feed** the output of task, along with the previous context to the LLM and perform reasoning task on context and output and identify which tasks are independent and can be done simultaneously to get the idea of the workspace.
5. Get the new tasks and **repeat** from step 3, until all tasks are finished.
6. Show the **final inference** of the session.

**Github Issue 2:** _One way prompt to generation_
Solution Algorithm (Simplified Version):
1. Take **User's Input**.
2. **Feed** it to LLM with the prompt and perform reasoning task on user's input and identify which tasks are independent and can be done simultaneously to get the idea of the workspace (spit out the tokens being generated for reasoning).
3. Get the **JSON** sequence of tasks and Proceed for those tasks **simultaneously**, and for file reading tasks to get just the context of each file, make use of LLM prepared one-time meta-data. (Give the user an opportunity to talk to the agent live, while LLM gives out some reasoning along with tasks, Nudge the user's query into the context by safely maintaing the correct structure of the context)
4. **Feed** the output of task, along with the previous context to the LLM and perform reasoning task on context and output and identify which tasks are independent and can be done simultaneously to get the idea of the workspace. (Show the reasoning and give the user, a chance again)
5. Get the new tasks and **repeat** from step 3, until all tasks are finished.
6. Show the **final inference** of the session.

# 🎉 Congratulations!! We are done with the **algorithmic part**, and we just need to translate it to Python code by handeling edge cases.